In [2]:
import polars as pl
from math import sqrt, log
from collections import defaultdict
from heapq import heapify, heappush, heappop,heapreplace
from tqdm import tqdm
import pickle

In [3]:
articles_path='../data/articles.parquet'
transaction_path='../data/transactions.parquet'

In [4]:
articles=pl.read_parquet(articles_path)
transactions=pl.read_parquet(transaction_path)

In [5]:
articles.shape

(105542, 12)

In [6]:
articles.head()

article_id,product_code,product_type_no,graphical_appearance_no,colour_group_code,perceived_colour_value_id,perceived_colour_master_id,department_no,index_code,index_group_no,section_no,garment_group_no
i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,i64
108775015,108775,253,1010016,9,4,5,1676,"""A""",1,16,1002
108775044,108775,253,1010016,10,3,9,1676,"""A""",1,16,1002
108775051,108775,253,1010017,11,1,9,1676,"""A""",1,16,1002
110065001,110065,306,1010016,9,4,5,1339,"""B""",1,61,1017
110065002,110065,306,1010016,10,3,9,1339,"""B""",1,61,1017


In [7]:
item_users_set=defaultdict(set)

In [8]:
transactions.shape

(31788324, 5)

In [9]:
transactions.head()

customer_id,article_id,price,sales_channel_id,time
str,i64,f64,u8,f64
"""000058a12d5b43e67d225668fa1f8d…",663713001,0.050831,2,1.5374e9
"""000058a12d5b43e67d225668fa1f8d…",541518023,0.030492,2,1.5374e9
"""00007d2de826758b65a93dd24ce629…",505221004,0.015237,2,1.5374e9
"""00007d2de826758b65a93dd24ce629…",685687003,0.016932,2,1.5374e9
"""00007d2de826758b65a93dd24ce629…",685687004,0.016932,2,1.5374e9


In [10]:
group_len=transactions.group_by('customer_id').len()['len'].to_numpy()

In [11]:
max(group_len)

np.uint32(1895)

In [12]:
group_len.shape[0]

1362281

In [13]:
sum(group_len==1)

np.int64(131514)

In [14]:
for i in range(5,100+1,10):
    print(f'users count times >= {i} :',sum(group_len>=i))

users count times >= 5 : 925558
users count times >= 15 : 534761
users count times >= 25 : 367845
users count times >= 35 : 267297
users count times >= 45 : 200804
users count times >= 55 : 154672
users count times >= 65 : 121143
users count times >= 75 : 96220
users count times >= 85 : 77342
users count times >= 95 : 63128


观察到有的用户交互次数太大，做相似度计算时过于耗费时间，因此选择截断，只保留最近的30个

In [15]:
def build_itemcf(transactions,topk=50):
    item_cnt = defaultdict(int)
    cooc = defaultdict(float)

    for _, g in tqdm(transactions.group_by("customer_id"),total=transactions['customer_id'].unique().shape[0]):
        g = g.sort("time", descending=True).head(25) # 只能最近的25个防止计算超出内存限制
        items = g.select(["article_id", "time"]).unique().to_numpy()
        for i, ti in items:
            item_cnt[i] += 1
            for j, tj in items:
                if i == j:
                    continue
                dt = abs(ti - tj) # 购买两个物品的间隔时间
                time_w = 1 / (1 + dt / (7 * 86400)) # 约束，在相邻时间内购买的物品，应有较大的权重
                cooc[(i, j)] += time_w

    # 用小根堆堆每个物品保留最相似的20个
    item_sim=defaultdict(list)
    for (i, j), cij in cooc.items():
        # 对热门物品打压
        weight=cij / (
            sqrt(item_cnt[i] * item_cnt[j]) * log(item_cnt[i] + 10)
        )

        if len(item_sim[i]) < topk:
            heappush(item_sim[i], (weight, j))
        else:
            if weight> item_sim[i][0][0]:
                heapreplace(item_sim[i], (weight, j))

    return item_sim

In [16]:
def recall_itemcf(data, item_sim, topk=50):
    dfs=[]
    DAY=86400

    for cid ,g in tqdm(data.group_by('customer_id'),total=data['customer_id'].unique().shape[0]):
        g=g.sort('time',descending=True)
        cid=cid[0]
        scores=defaultdict(float)

        max_time=g['time'].max()
        hist_items = set(g["article_id"])

        # 不能截断物品序列，因为之前交互过的物品中可能有相似度更大的临近物品
        for aid,t in zip(g['article_id'],g['time']):
            dt=(max_time-t)/DAY
            # 对用户的物品序列做权重，操作时间越近权重越大
            w_t=1/(1+dt)
            for w_sim,sim_item in item_sim[aid]:
                if sim_item in hist_items:
                    continue
                scores[sim_item]+=w_t*w_sim

        res=sorted(scores.items(),key=lambda x:x[1],reverse=True)[:topk]

        dfs.append(
            pl.DataFrame({
                "customer_id": [cid] * len(res),
                "article_id": [aid for aid, _ in res],
                "score": [score for _, score in res]
            })
        )

    return pl.concat(dfs)

In [17]:
def get_validation_data(data: pl.DataFrame):
    DAY = 86400
    WEEK = 7 * DAY

    max_time = data.select(pl.col("time").max()).item()

    valid_start = max_time - 6 * DAY
    train_start = valid_start - 6 * WEEK

    train_df = data.filter(
        (pl.col("time") >= train_start) &
        (pl.col("time") <  valid_start)
    )

    valid_df = data.filter(
        pl.col("time") >= valid_start
    )

    return train_df, valid_df

In [18]:
def metric_recall(data,topk=5):
    train_df,valid_df=get_validation_data(data)
    item_sim=build_itemcf(train_df)

    user_item = recall_itemcf(train_df, item_sim,topk * 10)

    pred_df = (
        user_item
        .group_by("customer_id")
        .agg(pl.col("article_id").alias("pred_items"))
    )

    true_df = (
        valid_df
    .group_by("customer_id")
        .agg(pl.col('article_id').unique().alias("true_items")))

    eval_df=pred_df.join(true_df,on='customer_id',how='inner')

    for k in range(10, topk * 10 + 1, 10):
        total = 0.0
        cnt = 0

        for pred, true in zip(eval_df["pred_items"], eval_df["true_items"]):
            if len(true) == 0:
                continue
            total += len(set(pred[:k]) & set(true)) / len(true)
            cnt += 1

        print(f"Recall@{k}: {total / cnt:.6f}")
    return item_sim

In [19]:
with open("../save/candidate/item_sim.pkl", "rb") as f:
    item_sim=pickle.load(f)


In [20]:
item_sim=metric_recall(transactions)

100%|██████████| 312215/312215 [02:24<00:00, 2156.08it/s]


Recall@10: 0.021665
Recall@20: 0.029376
Recall@30: 0.035229
Recall@40: 0.039371
Recall@50: 0.043304


In [20]:
res=recall_itemcf(transactions,item_sim)
res.write_parquet('itemcf_recall_100.parquet')